# Train Tokenizer

- using `rustbpe` tokenizer, equivalent to HF but simpler
- this should **exactly** match tokenizer produced by NanoChat

In [56]:
import datasets
import rustbpe
import tiktoken
import pickle

In [57]:
dataset = datasets.load_dataset(
    "HuggingFaceFW/fineweb-edu",
    name="sample-100BT",
    split="train",
)
dataset = dataset.shuffle(seed=42)  # Match nanochat repackage_data_reference.py seed

In [58]:
# For Testing
# max_chars = 2_000_000
# vocab_size = 1024
# doc_cap = 10000

# Match params used in nanochat speedrun.sh
# python -m scripts.tok_train --max_chars=2000000000 --vocab_size=65536
max_chars = 2000000000
vocab_size = 65536
doc_cap = 10000

# from nanochat tokenizer.py
# NOTE: this split pattern deviates from GPT-4 in that we use \p{N}{1,2} instead of \p{N}{1,3}
# I did this because I didn't want to "waste" too many tokens on numbers for smaller vocab sizes.
# I haven't validated that this is actually a good idea
SPLIT_PATTERN = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,2}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""

# from nanochat tokenizer.py
SPECIAL_TOKENS = [
    # every document begins with the Beginning of Sequence (BOS) token that delimits documents
    "<|bos|>",
    # tokens below are only used during finetuning to render Conversations into token ids
    "<|user_start|>", # user messages
    "<|user_end|>",
    "<|assistant_start|>", # assistant messages
    "<|assistant_end|>",
    "<|python_start|>", # assistant invokes python REPL tool
    "<|python_end|>",
    "<|output_start|>", # python REPL outputs back to assistant
    "<|output_end|>",
]

In [60]:
train_docs = []
char_count = 0
for i, example in enumerate(dataset):
    text = example["text"]
    if len(text) > doc_cap:
        text = text[:doc_cap]
    train_docs.append(text)
    char_count += len(text)
    
    if i % 100000 == 0 or char_count >= max_chars:
        pct = (char_count / max_chars) * 100
        print(f"Processed {char_count} / {max_chars} ({pct:.2f}%)")
    
    if char_count >= max_chars:
        break

Processed 8657 / 2000000000 (0.00%)
Processed 376164670 / 2000000000 (18.81%)
Processed 751674822 / 2000000000 (37.58%)
Processed 1126477659 / 2000000000 (56.32%)
Processed 1502706986 / 2000000000 (75.14%)
Processed 1877824553 / 2000000000 (93.89%)
Processed 2000002067 / 2000000000 (100.00%)


In [61]:
# Create tokenizer and train on your data
vocab_size_no_specials = vocab_size - len(SPECIAL_TOKENS)
tokenizer = rustbpe.Tokenizer()
tokenizer.train_from_iterator(
    train_docs,
    vocab_size=vocab_size_no_specials,
    pattern=SPLIT_PATTERN
)

In [62]:
pattern = tokenizer.get_pattern()
mergeable_ranks_list = tokenizer.get_mergeable_ranks()
mergeable_ranks = {bytes(k): v for k, v in mergeable_ranks_list}
tokens_offset = len(mergeable_ranks)
special_tokens = {name: tokens_offset + i for i, name in enumerate(SPECIAL_TOKENS)}
enc = tiktoken.Encoding(
    name="rustbpe",
    pat_str=pattern,
    mergeable_ranks=mergeable_ranks, # dict[bytes, int] (token bytes -> merge priority rank)
    special_tokens=special_tokens, # dict[str, int] (special token name -> token id)
)

In [63]:
with open("../data/tokenizer.pkl", "wb") as f:
    pickle.dump(enc, f)

In [ ]:
token_strings = [enc.decode([i]) for i in range(enc.n_vocab)]  # list[str] (token id -> token string)
token_bytes = []
for i in range(enc.n_vocab):
    tok_str = token_strings[i]
    if tok_str in special_tokens:
        # special tokens are not byte sequences
        token_bytes.append(0)
    else:
        token_bytes.append(len(tok_str.encode("utf-8")))

with open("../data/token_bytes.pkl", "wb") as f:
    pickle.dump(token_bytes, f)

# Validate against NanoChat

In [65]:
enc2 = pickle.load(open("/home/user/.cache/nanochat/tokenizer/tokenizer.pkl", "rb"))

In [71]:
print(enc2._pat_str)
enc2._pat_str == enc._pat_str

'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,2}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+


True

In [72]:
print(len(enc2._special_tokens))
enc2._special_tokens == enc._special_tokens

9


True

In [73]:
print(len(enc2._mergeable_ranks), len(enc._mergeable_ranks))
enc2._mergeable_ranks == enc._mergeable_ranks

65527 65527


True

In [74]:
import torch
token_bytes2 = torch.load("/home/user/.cache/nanochat/tokenizer/token_bytes.pt")
token_bytes2 = token_bytes2.tolist()

In [76]:
print(len(token_bytes2))
token_bytes2 == token_bytes

65536


True

In [77]:
for i, example in enumerate(dataset):
    tokens = enc.encode_ordinary(example["text"])
    tokens2 = enc2.encode_ordinary(example["text"])
    assert tokens2 == tokens
    if i > 1000:
        break
print(f"Checked tokenization consistency on {i+1} examples successfully.")

Checked tokenization consistency on 1002 examples successfully.


# Find Eval Set Start

In [1]:
import datasets
import pickle

/home/user/projects/my-nanochat/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset = datasets.load_dataset(
    "HuggingFaceFW/fineweb-edu",
    name="sample-100BT",
    split="train",
)
dataset = dataset.shuffle(seed=42)  # Match nanochat repackage_data_reference.py seed

In [3]:
# Tokenizer
tokenizer_path = "../data/tokenizer.pkl"
tokenizer = pickle.load(open(tokenizer_path, "rb"))

In [4]:
for i in range(1024*52, 1024*52+10):
    example = dataset[i]
    tokens = tokenizer.encode_ordinary(example["text"])
    test_seq = tokens[:20]
    
    print(test_seq)

[1711, 287, 1601, 355, 9115, 351, 57432, 10, 449, 2537, 2653, 3846, 261, 17084, 327, 257, 2678, 9912, 57432, 10]
[76, 30455, 3236, 44, 3409, 46, 32, 1328, 372, 85, 17867, 41, 2756, 334, 4490, 283, 5033, 287, 1205, 257]
[84, 14598, 5897, 471, 309, 6140, 414, 257, 3158, 1220, 261, 1159, 6494, 4757, 2012, 3863, 32, 49, 372, 7575]
[4425, 2268, 432, 3558, 33062, 4428, 2446, 3701, 283, 12352, 327, 4805, 9263, 4017, 5801, 44, 3622, 44, 288, 5182]
[1754, 345, 733, 871, 2393, 281, 5024, 4485, 355, 5024, 2570, 46, 1023, 261, 5024, 3305, 6635, 4816, 617, 356]
[13751, 5939, 58, 33619, 32, 50, 46, 52, 99, 3876, 4889, 567, 3156, 36771, 59, 44402, 44, 29362, 288, 12705]
[65, 1014, 4088, 309, 5493, 332, 261, 1132, 443, 4119, 327, 4512, 5655, 283, 7213, 44, 2009, 44, 1936, 44]
[2535, 331, 9849, 44, 261, 1083, 918, 384, 8453, 261, 13462, 1081, 31650, 13527, 47084, 44, 338, 883, 9402, 6301]
[2962, 9849, 44, 3528, 32, 2273, 261, 1716, 4658, 327, 261, 4725, 5939, 2430, 643, 747, 50874, 353, 257, 1614]
[86,

In [11]:
import os
import json
import pyarrow.parquet as pq

In [6]:
num_rg = dict()
base_path = "/home/user/.cache/nanochat/base_data"
filepaths = sorted(os.listdir(base_path))
start_idx = 0
for fp in filepaths:
    pf = pq.ParquetFile(os.path.join(base_path, fp))
    num_rg[fp] = {
        "num_row_groups": pf.num_row_groups,
        "start_idx": start_idx,
    }
    start_idx += pf.num_row_groups * 1024  # each row group has 1024 examples

In [7]:
# {'num_row_groups': 52, 'start_idx': 12736512}
# Meaning that eval set starts at dataset index 12736512
print(num_rg[filepaths[-1]])

{'num_row_groups': 52, 'start_idx': 12736512}


In [8]:
# Confirm
filename = "shard_00239.parquet"
pf = pq.ParquetFile(os.path.join(base_path, filename))
texts = pf.read_row_group(0).column('text').to_pylist()
tokens = tokenizer.encode_ordinary(texts[0])
dataset_idx = num_rg[filename]["start_idx"]    # first row group, first example
print(dataset_idx, tokens[:20])
print('---')
example = dataset[dataset_idx]
tokens = tokenizer.encode_ordinary(example["text"])
test_seq = tokens[:20]
print(dataset_idx, test_seq)

12736512 [449, 1083, 918, 19092, 12319, 23344, 18489, 32, 1185, 959, 408, 2351, 106, 587, 35632, 13551, 14806, 3750, 2514, 283]
---
12736512 [449, 1083, 918, 19092, 12319, 23344, 18489, 32, 1185, 959, 408, 2351, 106, 587, 35632, 13551, 14806, 3750, 2514, 283]


In [13]:
rowgroup_file = "../data/rowgroup_index.json"
with open(rowgroup_file, "w") as f:
    json.dump(num_rg, f, indent=4)